# Image Captioning
___
Generates and displays a COCO-style (Common Objects in Context) caption for the given image path.
    
Parameters:
- image_path (str): Path to the image file.
- model_id (str): Model ID to use for generating captions (default: "adept/fuyu-8b").
- device (str): Device to run the model on (default: "cuda"). Use "cpu" or "mps" for alternatives.

In [1]:
#Import Libraries
import os
import pickle
from sklearn.metrics.pairwise import cosine_similarity
import requests
import torch
import clip
from PIL import Image
from transformers import AutoTokenizer, FuyuForCausalLM, FuyuImageProcessor, FuyuProcessor

#### Code

In [2]:
def load_fuyu_model(model_id="adept/fuyu-8b", device="cuda"):
    dtype = torch.bfloat16 if device != "mps" else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = FuyuForCausalLM.from_pretrained(model_id, device_map=device, torch_dtype=dtype)
    processor = FuyuProcessor(image_processor=FuyuImageProcessor(), tokenizer=tokenizer)
    return tokenizer, model, processor

def generate_caption(image_path, tokenizer, model, processor, device="cuda"):
    #Prompt
    prompt = "Generate a coco-style caption.\n"
    image = Image.open(image_path)
    
    #Process inputs
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)
    
    #Generate caption
    generation_output = model.generate(**inputs, max_new_tokens=7)
    generation_text = processor.batch_decode(
        generation_output[:, -7:], skip_special_tokens=True
    )
    
    return generation_text[0]

#### Execute

In [3]:
#Example usage
device = "cuda"
model_id = "adept/fuyu-8b"

#Load the model, tokenizer, and processor once
tokenizer, model, processor = load_fuyu_model(model_id=model_id, device=device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
image_path = r"E:\OneDrive\Documents\007-Study Life\001-Urban Design\RC11_SkillsClasses\Workshop2\Datasets\Fossil\245-million-year-old reptile finally gets a name _ Natural History ....jpg"
caption = generate_caption(image_path, tokenizer, model, processor, device)
print("Generated Caption:", caption)

Setting `pad_token_id` to `eos_token_id`:71013 for open-end generation.
We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


Generated Caption: A collection of rocks and stones on display.


# Pairing
___
### Gathering Possible Keyword from Book Library
#### 1. Combine all processed keywords from Library

In [28]:
#Library Import
import os
import pickle
import torch
import clip
from PIL import Image

In [20]:
# Directory containing folders with pickle files
pickle_folder = r"E:\OneDrive\Documents\007-Study Life\001-Urban Design\RC11_SkillsClasses\Workshop1\Pickle"

# Collect all 'TEXT' values from pickle files
all_keywords = []
for root, dirs, files in os.walk(pickle_folder):
    for file_name in files:
        if file_name.endswith(".pkl"):
            file_path = os.path.join(root, file_name)
            if os.path.getsize(file_path) > 0:  # Ensure the file is not empty
                print(f"Processing file: {file_name} in folder: {root}")
                try:
                    with open(file_path, "rb") as f:
                        data = pickle.load(f)

                        # Handle different data structures
                        if isinstance(data, list):  # If it's a list
                            print(f"Data type: List with {len(data)} items.")
                            for item in data:
                                if isinstance(item, dict) and "KEYWORD" in item:
                                    all_keywords.append(item["KEYWORD"])
                        elif isinstance(data, dict):  # If it's a dictionary
                            print(f"Data type: Dictionary with keys: {list(data.keys())}")
                            if "KEYWORD" in data:
                                all_keywords.append(data["KEYWORD"])
                        else:
                            print("Unhandled data type.")
                except Exception as e:
                    print(f"Error loading {file_name}: {e}")

print(f"Total keywords collected: {len(all_keywords)}")

Processing file: Art and the Challenge of Market - Victoria D. Alexander_processed.pkl in folder: E:\OneDrive\Documents\007-Study Life\001-Urban Design\RC11_SkillsClasses\Workshop1\Pickle\Commodification
Data type: List with 2277 items.
Processing file: Babbling Corpse_ Vaporwave and - Grafton Tanner_processed.pkl in folder: E:\OneDrive\Documents\007-Study Life\001-Urban Design\RC11_SkillsClasses\Workshop1\Pickle\Commodification
Data type: List with 561 items.
Processing file: Babbling Corpse_ Vaporwave and the Commodi - Grafton Tanner_processed.pkl in folder: E:\OneDrive\Documents\007-Study Life\001-Urban Design\RC11_SkillsClasses\Workshop1\Pickle\Commodification
Data type: List with 561 items.
Processing file: Brand Islam_ The Marketing and - Faegheh Shirazi_processed.pkl in folder: E:\OneDrive\Documents\007-Study Life\001-Urban Design\RC11_SkillsClasses\Workshop1\Pickle\Commodification
Data type: List with 2099 items.
Processing file: Cannibal Culture_ Art, Appropri - Deborah Root_p

In [27]:
# Initialize an empty set to store unique words
unique_keywords = set()
for sentence in all_keywords:
    words = sentence.split() 
    unique_keywords.update(words)  

possible_keywords = list(unique_keywords)

#Testing
print(f"Total Keywords: \n{len(possible_keywords)}")
print(possible_keywords[20000:20100])

Total Keywords: 
40050
['pagina', 'rodham', 'urethra', 'dobbin', 'kra', 'jane', 'ail', 'protuber', 'feder', 'urbanist', 'sesquioxid', 'stagnat', 'argentin', 'bulletproof', 'avengingli', 'mening', 'dissatisfactori', 'caster', 'ventriculu', 'deadlin', 'parfait', 'hosti', 'finish', 'fiendlik', 'grim', 'postboy', 'cuticl', 'fussi', 'macaco', 'plutocrat', 'advantag', 'hippurit', 'cuckold', 'headband', 'miladi', 'thunder', 'thecla', 'gemmul', 'praxi', 'manism', 'vaunt', 'divu', 'goli', 'souther', 'acronym', 'callant', 'term', 'allegretto', 'silversmith', 'fract', 'tour', 'olona', 'costard', 'cloudlik', 'necker', 'plowman', 'sleek', 'unfear', 'planetari', 'adda', 'getaway', 'exhibitor', 'unenquir', 'graphit', 'stultifi', 'laten', 'unpurchas', 'honeydew', 'inexpi', 'hackney', 'purl', 'tronag', 'rhomboid', 'welder', 'colinear', 'paralysi', 'pebbl', 'unexpected', 'praemunir', 'churchmanship', 'liken', 'peda', 'declassifi', 'ingratitud', 'hypertens', 'spack', 'sewn', 'andron', 'andalusit', 'hedge

In [ ]:
#Save the possible_keywords to a pickle file named "All Keywords.pkl"
pickle_folder = "Pickle"
pickle_file = os.path.join(pickle_folder, "All Keywords.pkl")

with open(pickle_file, "wb") as f:
    pickle.dump(possible_keywords, f)

print(f"All Keywords have been saved to: {pickle_file}")

#### 2. Tokenize Keywords and Save to Pickle

In [32]:
#Load the model
clip_device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_preprocess = clip.load("ViT-B/32", device=clip_device)

#Create a folder to store pickle files if it doesn't exist
pickle_folder = "Pickle"
if not os.path.exists(pickle_folder):
    os.makedirs(pickle_folder)

#Define batch size (adjust based on your system's memory capacity)
batch_size = 200

#Process the keywords in batches
all_keywords_features = []
for i in range(0, len(possible_keywords), batch_size):
    batch_keywords = possible_keywords[i:i + batch_size]
    print(f"Processing batch {i // batch_size + 1}: {len(batch_keywords)} keywords")

    # Tokenize the batch of keywords
    text_tokens = clip.tokenize(batch_keywords).to(clip_device)

    # Encode the text features
    text_features = clip_model.encode_text(text_tokens).detach()

    # Normalize the text features
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # Collect features for all batches
    all_keywords_features.append(text_features)

#Concatenate all the tokenized features into a single tensor
all_keywords_features = torch.cat(all_keywords_features, dim=0)

#Save the tokenized features to a pickle file
#pickle_file = os.path.join(pickle_folder, "Keywords_Token.pkl")
#with open(pickle_file, "wb") as f:
#    pickle.dump(all_keywords_features.cpu(), f)  # Save on CPU to avoid CUDA dependency

print(f"Tokenized features saved to: {pickle_file}")

Processing batch 1: 200 keywords
Processing batch 2: 200 keywords
Processing batch 3: 200 keywords
Processing batch 4: 200 keywords
Processing batch 5: 200 keywords
Processing batch 6: 200 keywords
Processing batch 7: 200 keywords
Processing batch 8: 200 keywords
Processing batch 9: 200 keywords
Processing batch 10: 200 keywords
Processing batch 11: 200 keywords
Processing batch 12: 200 keywords
Processing batch 13: 200 keywords
Processing batch 14: 200 keywords
Processing batch 15: 200 keywords
Processing batch 16: 200 keywords
Processing batch 17: 200 keywords
Processing batch 18: 200 keywords
Processing batch 19: 200 keywords
Processing batch 20: 200 keywords
Processing batch 21: 200 keywords
Processing batch 22: 200 keywords
Processing batch 23: 200 keywords
Processing batch 24: 200 keywords
Processing batch 25: 200 keywords
Processing batch 26: 200 keywords
Processing batch 27: 200 keywords
Processing batch 28: 200 keywords
Processing batch 29: 200 keywords
Processing batch 30: 20

### Best Keywords to Image

In [37]:
def image_keywords(image_path, pickle_folder="Pickle", top_k=10):
    #Load the model
    clip_device = "cuda" if torch.cuda.is_available() else "cpu"
    clip_model, clip_preprocess = clip.load("ViT-B/32", device=clip_device)

    #Load and preprocess the image
    image = clip_preprocess(Image.open(image_path)).unsqueeze(0).to(clip_device)

    #Load the preprocessed text features from the pickle file
    pickle_file = os.path.join(pickle_folder, "Keywords_Token.pkl")
    with open(pickle_file, "rb") as f:
        all_keywords_features = pickle.load(f)

    #Ensure all_keywords_features are on the same device as the image features
    all_keywords_features = all_keywords_features.to(clip_device)
    image_features = clip_model.encode_image(image).detach()
    image_features /= image_features.norm(dim=-1, keepdim=True)

    #Calculate similarity between image and the pre-loaded keyword features
    similarities = (image_features @ all_keywords_features.T).squeeze(0)

    #Sort and retrieve the top `top_k` keywords
    top_indices = similarities.topk(top_k).indices
    best_keywords = [possible_keywords[idx] for idx in top_indices.cpu().numpy()]

    return best_keywords

In [40]:
image_path = r"E:\OneDrive\Documents\007-Study Life\001-Urban Design\RC11_SkillsClasses/Workshop2/Datasets/Fossil/Are Neanderthals the same species as us_ _ Natural History Museum.jpg"
image_caption = image_keywords(image_path)

print(f"10 Keywords of the Image:\n{image_caption}")

10 Keywords of the Image:
['hominid', 'mastoid', 'pterygoid', 'craniofaci', 'cranial', 'anthropogenesi', 'craniometri', 'scull', 'sphenoid', 'hominoid']
